Connecting NESO dataset

In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../database/wind_power.db")

df = pd.read_sql(
    "SELECT * FROM wind_generation_clean",
    conn
)

df.head()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,ND,TSD
0,01-JAN-2018,1,3064,5754,25593,26501
1,01-JAN-2018,2,3005,5754,26349,27207
2,01-JAN-2018,3,2950,5754,26240,27480
3,01-JAN-2018,4,2895,5754,25292,26760
4,01-JAN-2018,5,2892,5754,24416,26162


Creating Timestamp

- First we format SETTLEMENT_DATE from object to datatime format
- Then creating timestamp by combining SETTLEMENT_DATE and SETTLEMENT_PERIOD (where SETTLEMENT_PERIOD will be converted to time i.e. 1 = 00:00, 2 = 00:30)


In [14]:
df["SETTLEMENT_DATE"] = pd.to_datetime(
    df["SETTLEMENT_DATE"],
    format="mixed",
    dayfirst=True
)

In [15]:
df.head()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,ND,TSD,CAPACITY_FACTOR
0,2018-01-01,1,3064,5754,25593,26501,0.532499
1,2018-01-01,2,3005,5754,26349,27207,0.522245
2,2018-01-01,3,2950,5754,26240,27480,0.512687
3,2018-01-01,4,2895,5754,25292,26760,0.503128
4,2018-01-01,5,2892,5754,24416,26162,0.502607


In [16]:
df["SETTLEMENT_DATE"].dtype

dtype('<M8[ns]')

In [17]:
df["timestamp"] = (
    df["SETTLEMENT_DATE"]
    +
    pd.to_timedelta(
        (df["SETTLEMENT_PERIOD"] - 1) * 30,
        unit="m"
    )
)

In [21]:
df.head()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,ND,TSD,CAPACITY_FACTOR,timestamp
0,2018-01-01,1,3064,5754,25593,26501,0.532499,2018-01-01 00:00:00
1,2018-01-01,2,3005,5754,26349,27207,0.522245,2018-01-01 00:30:00
2,2018-01-01,3,2950,5754,26240,27480,0.512687,2018-01-01 01:00:00
3,2018-01-01,4,2895,5754,25292,26760,0.503128,2018-01-01 01:30:00
4,2018-01-01,5,2892,5754,24416,26162,0.502607,2018-01-01 02:00:00


In [22]:
df = df.sort_values(
    ["SETTLEMENT_DATE", "SETTLEMENT_PERIOD"]
)

In [24]:
df["EMBEDDED_WIND_GENERATION"].describe()

count    140256.000000
mean       1845.936409
std        1171.254858
min         125.000000
25%         910.000000
50%        1547.000000
75%        2551.000000
max        5962.000000
Name: EMBEDDED_WIND_GENERATION, dtype: float64

In [25]:
df["EMBEDDED_WIND_CAPACITY"].describe()

count    140256.000000
mean       6420.987059
std         241.605773
min        5754.000000
25%        6465.000000
50%        6527.000000
75%        6545.000000
max        6622.000000
Name: EMBEDDED_WIND_CAPACITY, dtype: float64

In [28]:
df["CAPACITY_FACTOR"] = (
    df["EMBEDDED_WIND_GENERATION"]
    /
    df["EMBEDDED_WIND_CAPACITY"]
)

In [29]:
df["CAPACITY_FACTOR"].describe()

count    140256.000000
mean          0.287462
std           0.181575
min           0.018922
25%           0.141600
50%           0.241213
75%           0.397820
max           0.900332
Name: CAPACITY_FACTOR, dtype: float64

In [30]:
df.to_parquet(
    "../data/interim/neso_processed/wind_generation_clean.prequet",#
    index=False
)

In [32]:
df.to_csv(
    "../data/interim/neso_processed/wind_generation_clean.csv",
    index = False
)

In [1]:
import pandas as pd

In [3]:
df = pd.read_parquet(
    "../data/interim/neso_processed/wind_generation_clean.prequet"
)

In [4]:
df.head()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,ND,TSD,CAPACITY_FACTOR,timestamp
0,2018-01-01,1,3064,5754,25593,26501,0.532499,2018-01-01 00:00:00
1,2018-01-01,2,3005,5754,26349,27207,0.522245,2018-01-01 00:30:00
2,2018-01-01,3,2950,5754,26240,27480,0.512687,2018-01-01 01:00:00
3,2018-01-01,4,2895,5754,25292,26760,0.503128,2018-01-01 01:30:00
4,2018-01-01,5,2892,5754,24416,26162,0.502607,2018-01-01 02:00:00


In [5]:
df["SETTLEMENT_PERIOD"].max()

50

In [6]:
df["SETTLEMENT_PERIOD"].value_counts().sort_index().tail()

SETTLEMENT_PERIOD
46    2922
47    2914
48    2914
49       8
50       8
Name: count, dtype: int64

In [7]:
df = df.sort_values(
    ["SETTLEMENT_DATE", "SETTLEMENT_PERIOD"]
)

In [13]:
d = df[
    [
        "SETTLEMENT_DATE",
        "EMBEDDED_WIND_GENERATION",
        "EMBEDDED_WIND_CAPACITY",
        "ND",
        "TSD",
        "CAPACITY_FACTOR",
        "timestamp"
    ]
]

In [15]:
d.to_parquet(
    "../data/interim/neso_processed/wind_generation_hourly.parquet",
    index=False
)

In [16]:
d.head()

,SETTLEMENT_DATE,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,ND,TSD,CAPACITY_FACTOR,timestamp
0,2018-01-01,3064,5754,25593,26501,0.532499,2018-01-01 00:00:00
1,2018-01-01,3005,5754,26349,27207,0.522245,2018-01-01 00:30:00
2,2018-01-01,2950,5754,26240,27480,0.512687,2018-01-01 01:00:00
3,2018-01-01,2895,5754,25292,26760,0.503128,2018-01-01 01:30:00
4,2018-01-01,2892,5754,24416,26162,0.502607,2018-01-01 02:00:00


Converison of 30 mins to hourly

In [19]:
df = df.sort_values(
    ["SETTLEMENT_DATE", "SETTLEMENT_PERIOD"]
)

In [21]:
hourly_df = (
    df.set_index("timestamp")
    .resample("H")
    .agg({
        "EMBEDDED_WIND_GENERATION":"mean",
        "EMBEDDED_WIND_CAPACITY":"mean",
        "CAPACITY_FACTOR":"mean",
        "ND":"mean",
        "TSD":"mean"
    }).reset_index()
)

C:\Users\LOQ\AppData\Local\Temp\ipykernel_26388\722793752.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample("H")


In [22]:
hourly_df.head()

,timestamp,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,CAPACITY_FACTOR,ND,TSD
0,2018-01-01 00:00:00,3034.5,5754.0,0.527372,25971.0,26854.0
1,2018-01-01 01:00:00,2922.5,5754.0,0.507908,25766.0,27120.0
2,2018-01-01 02:00:00,2891.0,5754.0,0.502433,24110.5,25954.0
3,2018-01-01 03:00:00,2803.0,5754.0,0.487139,22527.0,25020.0
4,2018-01-01 04:00:00,2638.5,5754.0,0.458551,21389.5,23867.5


In [23]:
hourly_df.to_parquet(
    "../data/interim/neso_processed/wind_generation_hourly.parquet",
    index=False
)

In [26]:
d1 = hourly_df.head(100)

In [30]:
d1.to_csv("../data/wind_generation_hourly.csv", index=False)

In [ ]:
d = pd.read_parquet(
    "../"
)